# SaaS Business Analytics — Data Quality & Cleaning

**Purpose:** build a reproducible, analyst-ready cleaning layer from the six raw SaaS datasets.

### Cleaning philosophy
- Raw CSVs are never overwritten.
- Cleaning should standardize data without silently deleting valid business records.
- Invalid relationships are investigated explicitly.
- Invalid subscription date sequences are **flagged**, not automatically deleted.
- Failed payments remain available for operational analysis but are excluded from collected-revenue KPIs.
- Refunded payments are represented as negative cash impact.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


## 1. Project paths

This notebook lives in `notebooks/`, so the project root is one level above it.


In [2]:
ROOT = Path("..")
RAW = ROOT / "data" / "raw"
CLEAN = ROOT / "data" / "cleaned"
DOCS = ROOT / "docs"

CLEAN.mkdir(parents=True, exist_ok=True)
DOCS.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT.resolve())
print("Raw data:", RAW.resolve())
print("Cleaned data:", CLEAN.resolve())


Project root: C:\Users\ponna\OneDrive\Documents\SaaS_Business_Analytics
Raw data: C:\Users\ponna\OneDrive\Documents\SaaS_Business_Analytics\data\raw
Cleaned data: C:\Users\ponna\OneDrive\Documents\SaaS_Business_Analytics\data\cleaned


## 2. Load raw datasets


In [3]:
customers_raw = pd.read_csv(RAW / "dim_customers.csv")
plans_raw = pd.read_csv(RAW / "dim_plans.csv")
subscriptions_raw = pd.read_csv(RAW / "fact_subscriptions.csv")
payments_raw = pd.read_csv(RAW / "fact_payments.csv")
usage_raw = pd.read_csv(RAW / "fact_usage.csv")
support_raw = pd.read_csv(RAW / "fact_support.csv")

raw = {
    "customers": customers_raw,
    "plans": plans_raw,
    "subscriptions": subscriptions_raw,
    "payments": payments_raw,
    "usage": usage_raw,
    "support": support_raw
}

for name, df in raw.items():
    print(f"{name:15} {len(df):,} rows x {len(df.columns)} columns")


customers       1,550 rows x 6 columns
plans           3 rows x 4 columns
subscriptions   1,500 rows x 6 columns
payments        3,063 rows x 6 columns
usage           24,756 rows x 6 columns
support         784 rows x 7 columns


## 3. Raw data quality profile


In [4]:
raw_profile = pd.DataFrame([
    {
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "duplicate_rows": int(df.duplicated().sum()),
        "missing_values": int(df.isna().sum().sum())
    }
    for name, df in raw.items()
])

raw_profile


,dataset,rows,columns,duplicate_rows,missing_values
0,customers,1550,6,50,593
1,plans,3,4,0,0
2,subscriptions,1500,6,0,1043
3,payments,3063,6,0,0
4,usage,24756,6,0,942
5,support,784,7,0,0


In [5]:
for name, df in raw.items():
    print("\n" + "=" * 70)
    print(name.upper())
    print("=" * 70)
    print(df.dtypes)
    print("\nMissing values:")
    print(df.isna().sum())



CUSTOMERS
customer_id     object
signup_date     object
country         object
industry        object
company_size    object
segment         object
dtype: object

Missing values:
customer_id       0
signup_date       0
country         348
industry        245
company_size      0
segment           0
dtype: int64

PLANS
plan_id           object
plan_name         object
monthly_price    float64
billing_cycle     object
dtype: object

Missing values:
plan_id          0
plan_name        0
monthly_price    0
billing_cycle    0
dtype: int64

SUBSCRIPTIONS
subscription_id    object
customer_id        object
plan_id            object
start_date         object
end_date           object
status             object
dtype: object

Missing values:
subscription_id       0
customer_id           0
plan_id               0
start_date            0
end_date           1043
status                0
dtype: int64

PAYMENTS
payment_id          object
subscription_id     object
payment_date        object
amount    

## 4. Business keys and duplicate checks


In [6]:
key_map = {
    "customers": "customer_id",
    "plans": "plan_id",
    "subscriptions": "subscription_id",
    "payments": "payment_id",
    "usage": "usage_id",
    "support": "ticket_id"
}

key_check = []

for name, key in key_map.items():
    df = raw[name]
    key_check.append({
        "dataset": name,
        "business_key": key,
        "duplicate_keys": int(df[key].astype("string").str.strip().duplicated().sum()),
        "missing_keys": int(df[key].isna().sum())
    })

key_check = pd.DataFrame(key_check)
key_check


,dataset,business_key,duplicate_keys,missing_keys
0,customers,customer_id,50,0
1,plans,plan_id,0,0
2,subscriptions,subscription_id,0,0
3,payments,payment_id,0,0
4,usage,usage_id,0,0
5,support,ticket_id,0,0


## 5. Clean customers

Customer IDs are the business key. The raw customer table contains duplicate customer records, so we keep one record per customer ID.

Categorical values are standardized so that equivalent labels do not create separate business segments.


In [7]:
customers = customers_raw.copy()

customers.columns = [c.strip() for c in customers.columns]
customers["customer_id"] = customers["customer_id"].astype("string").str.strip()
customers["signup_date"] = pd.to_datetime(customers["signup_date"], errors="coerce")

for col in ["country", "industry", "company_size", "segment"]:
    customers[col] = customers[col].astype("string").str.strip()

country_map = {
    "usa": "United States",
    "us": "United States",
    "united states": "United States",
    "canada": "Canada",
    "ca": "Canada",
    "uk": "United Kingdom",
    "united kingdom": "United Kingdom"
}

segment_map = {
    "mid": "Mid-Market",
    "mid-market": "Mid-Market",
    "smb": "SMB",
    "small business": "SMB",
    "ent": "Enterprise",
    "enterprise": "Enterprise"
}

customers["country"] = (
    customers["country"].str.lower().map(country_map)
    .fillna(customers["country"])
    .fillna("Unknown")
)

customers["industry"] = customers["industry"].fillna("Unknown").str.title()
customers["company_size"] = customers["company_size"].fillna("Unknown")
customers["segment"] = (
    customers["segment"].str.lower().map(segment_map)
    .fillna(customers["segment"])
    .fillna("Unknown")
)

customers = customers.drop_duplicates(subset="customer_id", keep="first").copy()

customers.columns = [
    "Customer_Id", "Signup_Date", "Country",
    "Industry", "Company_Size", "Segment"
]

print("Raw customer rows:", len(customers_raw))
print("Clean customer rows:", len(customers))
customers.head()


Raw customer rows: 1550
Clean customer rows: 1500


,Customer_Id,Signup_Date,Country,Industry,Company_Size,Segment
0,CUST_2000,2025-09-15,Unknown,Logistics,unknown,SMB
1,CUST_2001,2025-10-10,Unknown,E-Commerce,500+,Enterprise
2,CUST_2002,2025-03-19,Unknown,Logistics,201-500,SMB
3,CUST_2003,2025-11-30,United States,Finance,51-200,SMB
4,CUST_2004,2025-01-30,Unknown,Unknown,51-200,SMB


## 6. Clean plans


In [8]:
plans = plans_raw.copy()

plans.columns = [c.strip() for c in plans.columns]
plans["plan_id"] = plans["plan_id"].astype("string").str.strip().str.lower()
plans["plan_name"] = plans["plan_name"].astype("string").str.strip().str.title()
plans["monthly_price"] = pd.to_numeric(plans["monthly_price"], errors="coerce")
plans["billing_cycle"] = plans["billing_cycle"].astype("string").str.strip().str.title()

plans = plans.drop_duplicates(subset="plan_id", keep="first").copy()

plans.columns = [
    "Plan_Id", "Plan_Name", "Monthly_Price", "Billing_Cycle"
]

plans


,Plan_Id,Plan_Name,Monthly_Price,Billing_Cycle
0,p_basic,Basic,29.00,Monthly
1,p_pro,Pro,79.00,Monthly
2,p_ent,Enterprise,499.00,Annual


## 7. Clean subscriptions

**Important business rule:** all 1,500 subscription records are retained when their IDs and parent keys are valid.

Some synthetic records have an `end_date` earlier than `start_date`. Those records are **data-quality exceptions**. We flag them instead of deleting the subscription, because the subscription status and business key may still be useful for churn analysis.


In [9]:
subscriptions = subscriptions_raw.copy()

subscriptions.columns = [c.strip() for c in subscriptions.columns]

subscriptions["subscription_id"] = (
    subscriptions["subscription_id"].astype("string").str.strip()
)
subscriptions["customer_id"] = (
    subscriptions["customer_id"].astype("string").str.strip()
)
subscriptions["plan_id"] = (
    subscriptions["plan_id"].astype("string").str.strip().str.lower()
)

subscriptions["start_date"] = pd.to_datetime(
    subscriptions["start_date"], errors="coerce"
)
subscriptions["end_date"] = pd.to_datetime(
    subscriptions["end_date"], errors="coerce"
)

status_map = {
    "act": "Active",
    "active": "Active",
    "active_plan": "Active",
    "canceled": "Cancelled",
    "cancelled": "Cancelled",
    "churned": "Churned"
}

subscriptions["status"] = (
    subscriptions["status"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map(status_map)
)

subscriptions["Date_Quality_Flag"] = np.select(
    [
        subscriptions["start_date"].isna(),
        subscriptions["end_date"].notna()
        & (subscriptions["end_date"] < subscriptions["start_date"])
    ],
    [
        "Missing start date",
        "End date before start date"
    ],
    default="OK"
)

subscriptions["subscription_days"] = (
    subscriptions["end_date"] - subscriptions["start_date"]
).dt.days

subscriptions.loc[
    subscriptions["subscription_days"] < 0,
    "subscription_days"
] = np.nan

valid_customer = subscriptions["customer_id"].isin(customers["Customer_Id"])
valid_plan = subscriptions["plan_id"].isin(plans["Plan_Id"])
valid_id = subscriptions["subscription_id"].notna()
valid_status = subscriptions["status"].notna()

subscription_exclusions = pd.DataFrame({
    "Rule": [
        "Missing subscription ID",
        "Invalid customer ID",
        "Invalid plan ID",
        "Unrecognized status"
    ],
    "Excluded_Rows": [
        int((~valid_id).sum()),
        int((~valid_customer).sum()),
        int((~valid_plan).sum()),
        int((~valid_status).sum())
    ]
})

# Retain records with valid business keys and a recognized status.
subscriptions = subscriptions[
    valid_id & valid_customer & valid_plan & valid_status
].copy()

subscriptions.columns = [
    "Subscription_Id", "Customer_Id", "Plan_Id",
    "Start_Date", "End_Date", "Status",
    "Date_Quality_Flag", "Subscription_Days"
]

subscriptions["Subscription_Days"] = (
    pd.to_numeric(subscriptions["Subscription_Days"], errors="coerce")
    .astype("Int64")
)

print("Raw subscriptions:", len(subscriptions_raw))
print("Clean subscriptions:", len(subscriptions))
print("\nDate-quality exceptions:")
print(
    subscriptions["Date_Quality_Flag"]
    .value_counts(dropna=False)
)

subscription_exclusions


Raw subscriptions: 1500
Clean subscriptions: 1500

Date-quality exceptions:
Date_Quality_Flag
OK                            1477
End date before start date      23
Name: count, dtype: int64


,Rule,Excluded_Rows
0,Missing subscription ID,0
1,Invalid customer ID,0
2,Invalid plan ID,0
3,Unrecognized status,0


## 8. Investigate invalid subscription date sequences


In [10]:
invalid_subscription_dates = subscriptions[
    subscriptions["Date_Quality_Flag"] == "End date before start date"
].copy()

print("Invalid date-sequence records:", len(invalid_subscription_dates))

invalid_subscription_dates[
    [
        "Subscription_Id", "Customer_Id", "Plan_Id",
        "Start_Date", "End_Date", "Status"
    ]
]


Invalid date-sequence records: 23


,Subscription_Id,Customer_Id,Plan_Id,Start_Date,End_Date,Status
181,SUB_30181,CUST_2181,p_pro,2025-06-06,2025-05-29,Churned
307,SUB_30307,CUST_2307,p_basic,2025-12-15,2025-12-01,Cancelled
352,SUB_30352,CUST_2352,p_basic,2025-08-28,2025-08-09,Churned
424,SUB_30424,CUST_2424,p_pro,2025-12-13,2025-12-03,Cancelled
586,SUB_30586,CUST_2586,p_basic,2025-08-11,2025-07-28,Cancelled
613,SUB_30613,CUST_2613,p_ent,2025-11-21,2025-11-05,Cancelled
678,SUB_30678,CUST_2678,p_ent,2025-03-10,2025-02-25,Cancelled
778,SUB_30778,CUST_2778,p_pro,2025-01-03,2024-12-16,Cancelled
782,SUB_30782,CUST_2782,p_basic,2025-08-10,2025-07-21,Cancelled
858,SUB_30858,CUST_2858,p_basic,2025-05-24,2025-05-15,Cancelled


## 9. Clean payments

Payment records are retained when their payment ID, subscription ID, date, and amount are valid.

- `Paid` = collected revenue
- `Refunded` = cash returned to customer; represented as a negative amount
- `Failed` = unsuccessful payment; retained for payment-operation analysis but excluded from revenue KPIs


In [11]:
payments = payments_raw.copy()

payments.columns = [c.strip() for c in payments.columns]

payments["payment_id"] = payments["payment_id"].astype("string").str.strip()
payments["subscription_id"] = (
    payments["subscription_id"].astype("string").str.strip()
)
payments["payment_date"] = pd.to_datetime(
    payments["payment_date"], errors="coerce"
)
payments["amount"] = pd.to_numeric(
    payments["amount"], errors="coerce"
)
payments["payment_status"] = (
    payments["payment_status"].astype("string").str.strip().str.title()
)
payments["payment_method"] = (
    payments["payment_method"].astype("string").str.strip().str.upper()
)

# Store refunds as negative cash impact.
refund_mask = payments["payment_status"].eq("Refunded")
payments.loc[refund_mask, "amount"] = payments.loc[
    refund_mask, "amount"
].abs() * -1

valid_payment = (
    payments["payment_id"].notna()
    & payments["subscription_id"].isin(subscriptions["Subscription_Id"])
    & payments["payment_date"].notna()
    & payments["amount"].notna()
)

payment_exclusions = pd.DataFrame({
    "Rule": [
        "Missing payment ID",
        "Invalid subscription ID",
        "Missing payment date",
        "Missing amount"
    ],
    "Excluded_Rows": [
        int(payments["payment_id"].isna().sum()),
        int((~payments["subscription_id"].isin(
            subscriptions["Subscription_Id"]
        )).sum()),
        int(payments["payment_date"].isna().sum()),
        int(payments["amount"].isna().sum())
    ]
})

payments = payments[valid_payment].copy()

payments.columns = [
    "Payment_Id", "Subscription_Id", "Payment_Date",
    "Amount", "Payment_Status", "Payment_Method"
]

print("Raw payments:", len(payments_raw))
print("Clean payments:", len(payments))
print("\nPayment statuses:")
print(payments["Payment_Status"].value_counts())

payment_exclusions


Raw payments: 3063
Clean payments: 3063

Payment statuses:
Payment_Status
Paid        1542
Refunded     761
Failed       760
Name: count, dtype: Int64


,Rule,Excluded_Rows
0,Missing payment ID,0
1,Invalid subscription ID,0
2,Missing payment date,0
3,Missing amount,0


## 10. Clean product usage


In [12]:
usage = usage_raw.copy()

usage.columns = [c.strip() for c in usage.columns]

usage["usage_id"] = usage["usage_id"].astype("string").str.strip()
usage["customer_id"] = usage["customer_id"].astype("string").str.strip()
usage["event_date"] = pd.to_datetime(
    usage["event_date"], errors="coerce"
)
usage["feature_name"] = (
    usage["feature_name"].astype("string").str.strip().str.lower()
)
usage["feature_name"] = usage["feature_name"].fillna("Unknown")
usage["login_count"] = pd.to_numeric(
    usage["login_count"], errors="coerce"
)
usage["session_minutes"] = pd.to_numeric(
    usage["session_minutes"], errors="coerce"
)

usage["login_count"] = usage["login_count"].fillna(0).clip(lower=0)
usage["session_minutes"] = usage["session_minutes"].fillna(0).clip(lower=0)

valid_usage = (
    usage["usage_id"].notna()
    & usage["customer_id"].isin(customers["Customer_Id"])
    & usage["event_date"].notna()
)

usage = usage[valid_usage].copy()

usage.columns = [
    "Usage_Id", "Customer_Id", "Event_Date", "Feature_Name",
    "Login_Count", "Session_Minutes"
]

print("Raw usage:", len(usage_raw))
print("Clean usage:", len(usage))
usage.head()


Raw usage: 24756
Clean usage: 24756


,Usage_Id,Customer_Id,Event_Date,Feature_Name,Login_Count,Session_Minutes
0,USG_700001,CUST_2000,2025-09-08,dashboard_view,8,114.76
1,USG_700002,CUST_2000,2025-05-11,export_csv,3,44.68
2,USG_700003,CUST_2000,2025-01-17,export_csv,3,76.59
3,USG_700004,CUST_2000,2025-03-10,export_csv,8,58.81
4,USG_700005,CUST_2000,2025-05-25,export_csv,3,35.41


## 11. Clean support tickets


In [13]:
support = support_raw.copy()

support.columns = [c.strip() for c in support.columns]

support["ticket_id"] = support["ticket_id"].astype("string").str.strip()
support["customer_id"] = support["customer_id"].astype("string").str.strip()
support["ticket_date"] = pd.to_datetime(
    support["ticket_date"], errors="coerce"
)
support["priority"] = (
    support["priority"].astype("string").str.strip().str.title()
)
support["category"] = (
    support["category"].astype("string").str.strip().str.title()
)
support["resolution_time"] = pd.to_numeric(
    support["resolution_time"], errors="coerce"
)
support["status"] = (
    support["status"].astype("string").str.strip().str.title()
)

support["resolution_time"] = (
    support["resolution_time"].fillna(0).clip(lower=0)
)

valid_support = (
    support["ticket_id"].notna()
    & support["customer_id"].isin(customers["Customer_Id"])
    & support["ticket_date"].notna()
)

support = support[valid_support].copy()

support.columns = [
    "Ticket_Id", "Customer_Id", "Ticket_Date",
    "Priority", "Category", "Resolution_Time", "Status"
]

print("Raw support:", len(support_raw))
print("Clean support:", len(support))
support.head()


Raw support: 784
Clean support: 784


,Ticket_Id,Customer_Id,Ticket_Date,Priority,Category,Resolution_Time,Status
0,TCK_900001,CUST_2006,2025-02-17,High,Login Issue,12,Closed
1,TCK_900002,CUST_2006,2025-07-17,Low,Api Down,41,Resolved
2,TCK_900003,CUST_2007,2025-08-23,Low,Login Issue,21,Closed
3,TCK_900004,CUST_2008,2025-11-10,Medium,Login Issue,43,Resolved
4,TCK_900005,CUST_2008,2025-07-22,Medium,Feature Request,1,Resolved


## 12. Referential-integrity validation

The cleaned analytical layer should not contain orphan foreign keys.


In [14]:
integrity = pd.Series({
    "subscription_customer_orphans":
        (~subscriptions["Customer_Id"].isin(customers["Customer_Id"])).sum(),
    "subscription_plan_orphans":
        (~subscriptions["Plan_Id"].isin(plans["Plan_Id"])).sum(),
    "payment_subscription_orphans":
        (~payments["Subscription_Id"].isin(
            subscriptions["Subscription_Id"]
        )).sum(),
    "usage_customer_orphans":
        (~usage["Customer_Id"].isin(customers["Customer_Id"])).sum(),
    "support_customer_orphans":
        (~support["Customer_Id"].isin(customers["Customer_Id"])).sum()
}, name="orphan_rows")

integrity


subscription_customer_orphans    0
subscription_plan_orphans        0
payment_subscription_orphans     0
usage_customer_orphans           0
support_customer_orphans         0
Name: orphan_rows, dtype: int64

## 13. Final cleaned-data profile


In [15]:
cleaned = {
    "customers": customers,
    "plans": plans,
    "subscriptions": subscriptions,
    "payments": payments,
    "usage": usage,
    "support": support
}

cleaned_profile = pd.DataFrame([
    {
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "duplicate_rows": int(df.duplicated().sum()),
        "missing_values": int(df.isna().sum().sum())
    }
    for name, df in cleaned.items()
])

cleaned_profile


,dataset,rows,columns,duplicate_rows,missing_values
0,customers,1500,6,0,0
1,plans,3,4,0,0
2,subscriptions,1500,8,0,2109
3,payments,3063,6,0,0
4,usage,24756,6,0,0
5,support,784,7,0,0


## 14. Save the cleaned analytical layer

These exports are the inputs for the EDA, SQL, Excel, and Power BI stages.


In [16]:
output_files = {
    "customers": "customers_clean.csv",
    "plans": "plans_clean.csv",
    "subscriptions": "subscriptions_clean.csv",
    "payments": "payments_clean.csv",
    "usage": "usage_clean.csv",
    "support": "support_clean.csv"
}

for name, filename in output_files.items():
    cleaned[name].to_csv(CLEAN / filename, index=False)
    print(f"Saved: {filename}")


Saved: customers_clean.csv
Saved: plans_clean.csv
Saved: subscriptions_clean.csv
Saved: payments_clean.csv
Saved: usage_clean.csv
Saved: support_clean.csv


## 15. Reload the saved files

Reloading proves that the analytical layer exists independently of the notebook's memory.


In [17]:
saved = {
    name: pd.read_csv(CLEAN / filename)
    for name, filename in output_files.items()
}

for name, df in saved.items():
    print(f"{name:15} {len(df):,} rows x {len(df.columns)} columns")


customers       1,500 rows x 6 columns
plans           3 rows x 4 columns
subscriptions   1,500 rows x 8 columns
payments        3,063 rows x 6 columns
usage           24,756 rows x 6 columns
support         784 rows x 7 columns


## 16. Revenue and subscription KPI sanity check

**Important:** failed payments are not revenue. Net collected revenue is paid revenue plus the negative refund amounts.


In [18]:
paid_revenue = payments.loc[
    payments["Payment_Status"].eq("Paid"),
    "Amount"
].sum()

refund_value = payments.loc[
    payments["Payment_Status"].eq("Refunded"),
    "Amount"
].abs().sum()

net_collected_revenue = payments.loc[
    payments["Payment_Status"].isin(["Paid", "Refunded"]),
    "Amount"
].sum()

active_subscriptions = subscriptions["Status"].eq("Active").sum()
churned_subscriptions = subscriptions["Status"].eq("Churned").sum()
observed_churn_rate = (
    churned_subscriptions / len(subscriptions) * 100
)

kpi_snapshot = pd.Series({
    "Paid Revenue": paid_revenue,
    "Refund Value": refund_value,
    "Net Collected Revenue": net_collected_revenue,
    "Active Subscriptions": active_subscriptions,
    "Churned Subscriptions": churned_subscriptions,
    "Observed Churn Rate %": observed_churn_rate
})

kpi_snapshot.round(2)


Paid Revenue            258,370.00
Refund Value            135,639.00
Net Collected Revenue   122,731.00
Active Subscriptions      1,043.00
Churned Subscriptions       151.00
Observed Churn Rate %        10.07
dtype: float64

## 17. Save data-quality evidence


In [19]:
raw_profile.to_csv(
    DOCS / "raw_data_quality_summary.csv",
    index=False
)

cleaned_profile.to_csv(
    DOCS / "cleaned_data_quality_summary.csv",
    index=False
)

key_check.to_csv(
    DOCS / "business_key_check.csv",
    index=False
)

subscription_exclusions.to_csv(
    DOCS / "subscription_exclusion_rules.csv",
    index=False
)

payment_exclusions.to_csv(
    DOCS / "payment_exclusion_rules.csv",
    index=False
)

invalid_subscription_dates.to_csv(
    DOCS / "invalid_subscription_dates.csv",
    index=False
)

pd.DataFrame({
    "metric": kpi_snapshot.index,
    "value": kpi_snapshot.values
}).to_csv(
    DOCS / "kpi_validation_snapshot.csv",
    index=False
)

print("Data-quality evidence saved to docs/.")


Data-quality evidence saved to docs/.


# Final validation checklist

Before moving to EDA, verify:

- Raw data remains unchanged.
- Customer duplicate IDs were resolved.
- IDs and categorical values were standardized.
- Dates were converted to datetime.
- Subscription records with invalid date sequences were **flagged rather than silently deleted**.
- Failed payments remain in the payment table but are excluded from revenue KPIs.
- Refunded payments are negative cash impact.
- Foreign-key orphan counts are zero.
- Cleaned CSVs were saved and successfully reloaded.
- KPI sanity check completed.

### Next stage

`SaaS Business EDA` will use the cleaned analytical layer to answer business questions about:

**Revenue → Customers → Subscriptions → Churn → Product Usage → Support.**
